# inference-2B-1.ipynb - Semantic Few-Shot Retrieval (Qwen3-1.7B)

  uses `Qwen3-1.7B` adapter (attempt2_1p7b)

- FAISS semantic retrieval (`all-MiniLM-L6-v2` + top-5 examples)
- No quantization — 1.7B in BF16 (~3.4GB VRAM)
- Larger inference batch (16) for faster throughput
- 3-pass retry + fallback logic

In [ ]:
'''
Mount Google Drive to access datasets and save model checkpoints. We use the Colab-specific drive.mount function to mount the drive at /content/drive. After mounting, we print a confirmation message. This allows us to read our training data from Drive and also save our fine-tuned model checkpoints back to Drive for later use.
'''
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

%pip install -q "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
%pip install -q transformers peft accelerate bitsandbytes pandas lxml
%pip install -q sentence-transformers faiss-cpu

Mounted at /content/drive
Drive mounted.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.9 MB/s eta 0:00:00


In [ ]:
# ── INF_CONFIG ────────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/DL/midterm"

INF_CONFIG = {
    "base_model":            "Qwen/Qwen3-1.7B",
    "adapter_path":          f"{DRIVE_ROOT}/attempt2_1p7b",
    "train_csv_path":        f"{DRIVE_ROOT}/dl-spring-2026-svg-generation/train.csv",
    "test_csv_path":         f"{DRIVE_ROOT}/dl-spring-2026-svg-generation/test.csv",
    "submission_path":       f"{DRIVE_ROOT}/inference_2B_1_submission.csv",
    "max_seq_length":        4096,
    "max_new_tokens":        512,
    "infer_batch":           16,
    "temperature":           0.3,
    "top_p":                 0.9,
    "repetition_penalty":    1.1,
    # Few-shot retrieval
    "n_shots":               5,
    "encoder_model":         "all-MiniLM-L6-v2",   # 80MB, CPU-friendly
    "max_example_svg_chars": 600,
    "min_example_svg_chars": 100,
}

SEED = 42
print("INF_CONFIG:")
for k, v in INF_CONFIG.items():
    print(f"  {k}: {v}")

INF_CONFIG:
  base_model: Qwen/Qwen3-1.7B
  adapter_path: /content/drive/MyDrive/DL/midterm/attempt2_1p7b
  train_csv_path: /content/drive/MyDrive/DL/midterm/dl-spring-2026-svg-generation/train.csv
  test_csv_path: /content/drive/MyDrive/DL/midterm/dl-spring-2026-svg-generation/test.csv
  submission_path: /content/drive/MyDrive/DL/midterm/inference_2B_1_submission.csv
  max_seq_length: 4096
  max_new_tokens: 512
  infer_batch: 16
  temperature: 0.3
  top_p: 0.9
  repetition_penalty: 1.1
  n_shots: 5
  encoder_model: all-MiniLM-L6-v2
  max_example_svg_chars: 600
  min_example_svg_chars: 100


In [ ]:
'''
Set random seeds and device configuration. We set the random seed for both Python's random module and NumPy to ensure reproducibility of our results. We also check if a GPU is available using torch.cuda.is_available() and set the DEVICE variable accordingly. If a GPU is available, we print its name and VRAM capacity. This helps us confirm that we have the necessary hardware to fine-tune our model efficiently.
'''
import os, re, csv, time, random, json, pathlib, shutil, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from IPython.display import display, HTML

random.seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : NVIDIA L4
VRAM   : 23.7 GB


In [ ]:
'''
Verify Flash Attention 2 is active. We check three things: (1) the flash_attn package is installed, (2) the model's config indicates that flash_attention_2 is being used, and (3) the class name of the first attention layer contains "flash". This helps ensure that we're actually using Flash Attention 2, which is crucial for fitting the model in VRAM and achieving good performance. Flash attention 2 over traditional attention gave approx 5x speedup, without it iterattion were 0.03 iter/s, with flash attention 2 it is 0.17 iter/s on same hardware.
'''

print(f"Loading base model: {INF_CONFIG['base_model']} …")
_base = AutoModelForCausalLM.from_pretrained(
    INF_CONFIG["base_model"],
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

print(f"Attaching adapter: {INF_CONFIG['adapter_path']} …")
model = PeftModel.from_pretrained(_base, INF_CONFIG["adapter_path"])
model.eval()

tokenizer = AutoTokenizer.from_pretrained(INF_CONFIG["adapter_path"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_acfg = pathlib.Path(INF_CONFIG["adapter_path"]) / "adapter_config.json"
if _acfg.exists():
    _d = json.loads(_acfg.read_text())
    print(f"adapter base: {_d.get('base_model_name_or_path')}  r={_d.get('r')}")

print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading base model: Qwen/Qwen3-1.7B …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Attaching adapter: /content/drive/MyDrive/DL/midterm/attempt2_1p7b …
adapter base: Qwen/Qwen3-1.7B  r=32
Memory: 4.20 GB


In [ ]:
# ── Build semantic retriever (sentence-transformers + FAISS) ──────────────────
# Encodes all training prompts with all-MiniLM-L6-v2 (80MB, CPU).
# One-time cost ~3-5 min. FAISS search is <1ms per query.
'''
Few shot retrieval setup. We load the training data from the specified CSV file, filter it to create a retrieval pool of valid SVG examples within a certain character length range, and then encode the prompts using a sentence-transformer model. Finally, we build a FAISS index for efficient similarity search during inference. This allows us to retrieve relevant examples from the training set to use as few-shot demonstrations when generating SVGs for test prompts.
'''

_src = pathlib.Path(INF_CONFIG["train_csv_path"])
_dst = pathlib.Path("/content/train_local.csv")
if not _dst.exists():
    print("Copying train.csv locally …")
    shutil.copy2(_src, _dst)

print("Loading training data …")
with open(_dst, "r", encoding="utf-8", newline="", errors="replace") as f:
    rows = list(csv.DictReader(f))
train_df = pd.DataFrame(rows)[["prompt", "svg"]].dropna()

def _is_valid_xml(svg):
    try: ET.fromstring(svg); return True
    except: return False

lo = INF_CONFIG["min_example_svg_chars"]
hi = INF_CONFIG["max_example_svg_chars"]
mask = (
    train_df["svg"].str.startswith("<svg") &
    train_df["svg"].str.len().between(lo, hi) &
    train_df["svg"].apply(_is_valid_xml)
)
retrieval_df = train_df[mask].reset_index(drop=True)
print(f"Training rows     : {len(train_df):,}")
print(f"Retrieval pool    : {len(retrieval_df):,}  (svg {lo}–{hi} chars, valid XML)")

RETRIEVAL_PROMPTS = retrieval_df["prompt"].tolist()
RETRIEVAL_SVGS    = retrieval_df["svg"].tolist()

# ── Encode with sentence-transformer ─────────────────────────────────────────
print(f"\nLoading encoder: {INF_CONFIG['encoder_model']} …")
encoder = SentenceTransformer(INF_CONFIG["encoder_model"], device="cpu")

print("Encoding retrieval pool (this takes ~3-5 min on CPU) …")
embeddings = encoder.encode(
    RETRIEVAL_PROMPTS,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-norm so inner product = cosine similarity
).astype(np.float32)

# ── Build FAISS flat index ────────────────────────────────────────────────────
dim   = embeddings.shape[1]      # 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatIP(dim)   # exact inner product (= cosine after normalization)
index.add(embeddings)
print(f"FAISS index: {index.ntotal:,} vectors, dim={dim}")
print("Semantic retriever ready.")

Copying train.csv locally …
Loading training data …
Training rows     : 50,000
Retrieval pool    : 4,913  (svg 100–600 chars, valid XML)

Loading encoder: all-MiniLM-L6-v2 …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding retrieval pool (this takes ~3-5 min on CPU) …


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

FAISS index: 4,913 vectors, dim=384
Semantic retriever ready.


In [ ]:
# ── SVG helpers + few-shot chat formatter ─────────────────────────────────────
'''
Helper functions for SVG generation and few-shot chat formatting. We define a system prompt that instructs the model to generate compact, valid SVG markup based on user requests. We also define regex patterns for extracting SVG code and "think" sections from the model's output. We maintain a set of CSS color names and a mapping of shape keywords to SVG templates. The helper functions include extracting colors from prompts, checking if an SVG contains a specific color, generating fallback SVGs based on prompt keywords, extracting SVG code from model output, validating SVG structure, repairing common SVG issues, retrieving semantically similar examples for few-shot prompting, and running batched generation with retries.
'''
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return ONLY the SVG code — no explanation, no markdown fences. "
    'Use a single root <svg> element with xmlns=\"http://www.w3.org/2000/svg\" '
    'and viewBox=\"0 0 200 200\". Keep output under 15000 characters. '
    "Use EXACTLY the colors and shapes described in the prompt. "
    "Study the examples carefully and produce SVGs of similar complexity and structure."
)

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)
_THINK_RE = re.compile(r"<think>[\s\S]*?</think>", re.IGNORECASE)

_CSS_COLORS = {
    "red", "blue", "green", "yellow", "orange", "purple", "pink", "brown",
    "black", "white", "gray", "grey", "cyan", "magenta", "lime", "teal",
    "navy", "gold", "silver", "violet", "indigo", "maroon", "olive",
    "coral", "turquoise", "beige", "tan", "crimson", "aqua",
}
_SHAPE_SVG = {
    "circle":    lambda c: f'<circle cx="100" cy="100" r="70" fill="{c}"/>',
    "square":    lambda c: f'<rect x="30" y="30" width="140" height="140" fill="{c}"/>',
    "rectangle": lambda c: f'<rect x="20" y="50" width="160" height="100" fill="{c}"/>',
    "triangle":  lambda c: f'<polygon points="100,20 180,180 20,180" fill="{c}"/>',
    "star":      lambda c: f'<polygon points="100,15 120,70 180,70 132,107 150,165 100,128 50,165 68,107 20,70 80,70" fill="{c}"/>',
    "heart":     lambda c: f'<path d="M100,160 C60,130 10,100 10,60 A40,40 0 0,1 100,40 A40,40 0 0,1 190,60 C190,100 140,130 100,160Z" fill="{c}"/>',
}

def _extract_color(prompt):
    for w in re.findall(r'\b\w+\b', prompt.lower()):
        if w in _CSS_COLORS: return w
    return None

def _svg_has_color(svg, color):
    return color.lower() in svg.lower()

def fallback_svg(prompt):
    color = _extract_color(prompt) or "steelblue"
    inner = next(
        (fn(color) for kw, fn in _SHAPE_SVG.items() if kw in prompt.lower()),
        f'<circle cx="100" cy="100" r="70" fill="{color}"/>',
    )
    bg = "white" if color != "white" else "#f0f0f0"
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="200" height="200" viewBox="0 0 200 200">'
        f'<rect width="200" height="200" fill="{bg}"/>'
        + inner + '</svg>'
    )

def extract_svg(text):
    text = _THINK_RE.sub("", text)
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def is_valid_svg(svg):
    if not svg: return False
    try: return ET.fromstring(svg).tag.endswith("svg")
    except ET.ParseError: return False

def repair_svg(svg):
    if not svg: return svg
    if "xmlns" not in svg:
        svg = svg.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if "viewBox" not in svg and "viewbox" not in svg.lower():
        svg = svg.replace("<svg", '<svg viewBox="0 0 200 200"', 1)
    return svg

# ── Semantic retrieval via FAISS ──────────────────────────────────────────────
def get_examples(prompt, n=None):
    """Return top-n semantically similar (prompt, svg) training pairs."""
    n = n or INF_CONFIG["n_shots"]
    vec = encoder.encode([prompt], normalize_embeddings=True).astype(np.float32)
    _, idxs = index.search(vec, n)
    return [(RETRIEVAL_PROMPTS[i], RETRIEVAL_SVGS[i]) for i in idxs[0]]

def _make_chat(prompt):
    """Few-shot chat: inject top-n semantically similar training examples."""
    examples = get_examples(prompt)
    text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    for ex_prompt, ex_svg in examples:
        text += f"<|im_start|>user\n{ex_prompt} /no_think<|im_end|>\n"
        text += f"<|im_start|>assistant\n{ex_svg}<|im_end|>\n"
    text += f"<|im_start|>user\n{prompt} /no_think<|im_end|>\n"
    text += "<|im_start|>assistant\n"
    return text

# ── Generation helpers ────────────────────────────────────────────────────────
def _run_gen(texts):
    tokenizer.padding_side = "left"
    inp = tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=INF_CONFIG["max_seq_length"]
    ).to(DEVICE)
    plen = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=INF_CONFIG["max_new_tokens"],
            do_sample=True,
            temperature=INF_CONFIG["temperature"],
            top_p=INF_CONFIG["top_p"],
            repetition_penalty=INF_CONFIG["repetition_penalty"],
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [tokenizer.decode(o[plen:], skip_special_tokens=True) for o in out]

def _batch_gen(texts, batch_size):
    results = []
    for i in range(0, len(texts), batch_size):
        results.extend(_run_gen(texts[i : i + batch_size]))
    return results

def _parse_batch(prompts, raws):
    svgs, bad = [], []
    for j, (p, raw) in enumerate(zip(prompts, raws)):
        svg   = repair_svg(extract_svg(raw))
        color = _extract_color(p)
        ok    = is_valid_svg(svg) and (not color or _svg_has_color(svg, color))
        svgs.append(svg if ok else None)
        if not ok: bad.append(j)
    return svgs, bad

def generate_svg_batch(prompts, batch_size=None):
    """
    Three-pass batched retry:
      Pass 1 (fine-tuned + semantic few-shot): all prompts.
      Pass 2 (fine-tuned + semantic few-shot): failures from pass 1.
      Pass 3 (base model + semantic few-shot): failures from pass 2.
      Final: template fallback_svg().
    """
    if batch_size is None:
        batch_size = INF_CONFIG["infer_batch"]
    texts = [_make_chat(p) for p in prompts]

    model.enable_adapter_layers()
    svgs, bad1 = _parse_batch(prompts, _batch_gen(texts, batch_size))

    if bad1:
        raw2 = _batch_gen([texts[j] for j in bad1], batch_size)
        for j, raw in zip(bad1, raw2):
            svg2 = repair_svg(extract_svg(raw))
            if is_valid_svg(svg2):
                color = _extract_color(prompts[j])
                if not color or _svg_has_color(svg2, color):
                    svgs[j] = svg2

    bad2 = [j for j in bad1 if svgs[j] is None]
    if bad2:
        model.disable_adapter_layers()
        raw3 = _batch_gen([texts[j] for j in bad2], batch_size)
        model.enable_adapter_layers()
        for j, raw in zip(bad2, raw3):
            svg3 = repair_svg(extract_svg(raw))
            svgs[j] = svg3 if is_valid_svg(svg3) else fallback_svg(prompts[j])

    svgs = [s if s is not None else fallback_svg(p) for s, p in zip(svgs, prompts)]
    return svgs

print("Helpers + semantic retriever ready.")

Helpers + semantic retriever ready.


In [ ]:
# ── Smoke test: retrieved examples + generated output ─────────────────────────
# Single pass, max_new_tokens=512, no retry — fast visual check (~2 min).
'''
Smoke test generation. We take a few random prompts from the test set and add some simple shape prompts. We then generate SVGs for these prompts using the fine-tuned model with semantic few-shot retrieval, but we only do a single pass with max_new_tokens=512 and no retries. This allows us to quickly check if the model is producing valid SVG outputs that contain the expected colors and shapes based on the prompts and retrieved examples. We display the results in an HTML table format, showing the retrieved examples and the generated SVG side by side for each prompt, along with validity and color checks.
'''
_test_df      = pd.read_csv(INF_CONFIG["test_csv_path"])
_rand3        = _test_df.sample(n=3, random_state=SEED)["prompt"].tolist()
SMOKE_PROMPTS = _rand3 + ["a simple red circle", "a yellow star"]

def _smoke_gen(prompts_list):
    """Single-pass, no retry, 512 tokens — fast visual check only."""
    tokenizer.padding_side = "left"
    texts = [_make_chat(p) for p in prompts_list]
    inp   = tokenizer(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=INF_CONFIG["max_seq_length"]).to(DEVICE)
    plen  = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=512, do_sample=True,
            temperature=0.3, top_p=0.9, repetition_penalty=1.1,
            use_cache=True, pad_token_id=tokenizer.eos_token_id,
        )
    return [
        repair_svg(extract_svg(tokenizer.decode(o[plen:], skip_special_tokens=True))) or fallback_svg(p)
        for o, p in zip(out, prompts_list)
    ]

print("Generating smoke test (semantic few-shot, single pass) …")
model.enable_adapter_layers()
ft_svgs = _smoke_gen(SMOKE_PROMPTS)

for prompt, svg in zip(SMOKE_PROMPTS, ft_svgs):
    examples = get_examples(prompt)
    ex_cells = "".join(
        f'<td style="padding:6px; text-align:center; vertical-align:top;">'
        f'<div style="font-size:10px; max-width:130px; margin-bottom:4px;">{ep[:70]}…</div>'
        f'<div style="border:1px solid #ccc; width:120px; height:120px; display:inline-block;">{es}</div>'
        f'</td>'
        for ep, es in examples
    )
    valid  = is_valid_svg(svg)
    border = "2px solid green" if valid else "2px solid red"
    color  = _extract_color(prompt)
    c_lbl  = f"color={color} {'✓' if color and _svg_has_color(svg, color) else '✗'}" if color else ""
    result = (
        f'<td style="padding:6px; text-align:center; vertical-align:top;">'
        f'<div style="border:{border}; width:150px; height:150px; display:inline-block;">{svg}</div>'
        f'<div style="font-size:10px; margin-top:4px;">{"✓" if valid else "✗"} valid<br>{c_lbl}</div>'
        f'</td>'
    )
    display(HTML(
        f'<div style="margin-bottom:20px;">'
        f'<b style="font-size:12px;">{prompt}</b><br>'
        f'<table border="1" cellspacing="0" style="border-collapse:collapse; margin-top:6px;">'
        f'<tr><th colspan="{len(examples)}">Retrieved examples (semantic)</th><th>Generated (fine-tuned)</th></tr>'
        f'<tr>{ex_cells}{result}</tr>'
        f'</table></div>'
    ))

valid_count = sum(is_valid_svg(s) for s in ft_svgs)
print(f"Valid: {valid_count}/{len(SMOKE_PROMPTS)}  (smoke: single pass, max_new_tokens=512)")

Generating smoke test (semantic few-shot, single pass) …


Valid: 5/5  (smoke: single pass, max_new_tokens=512)


In [ ]:
# ── Generate submission CSV with checkpointing (10 chunks × 100 prompts) ──────
# Set MY_CHUNKS per Colab session. Re-run to resume from crashes.
#
# Example split across 10 Colabs (one chunk each):
#   Colab 0: MY_CHUNKS = [0]
#   Colab 1: MY_CHUNKS = [1]  ... etc.

'''
The fallback Logic was taking lots of time, so we switched to a chunked approach with retries. We split the test prompts into 10 chunks of 100 prompts each. Each Colab session can be assigned one or more chunks to process by setting the MY_CHUNKS variable. For each chunk, we check if the output CSV already exists and has the expected number of rows; if so, we skip that chunk (allowing for easy resumption after crashes). For each prompt in the chunk, we generate an SVG using the three-pass retry logic: first with the fine-tuned model and semantic few-shot retrieval, then retrying failures with the same setup, and finally falling back to the base model and ultimately a template SVG if all else fails. After processing all chunks, we check that all chunks are complete before merging them into the final submission CSV.
'''

MY_CHUNKS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]    

test_df    = pd.read_csv(INF_CONFIG["test_csv_path"])
prompts    = test_df["prompt"].tolist()
ids        = test_df["id"].tolist()

N_CHUNKS   = 10
CHUNK_SIZE = len(prompts) // N_CHUNKS
BATCH      = INF_CONFIG["infer_batch"]
CHUNK_DIR  = f"{DRIVE_ROOT}/inference_2B_1_chunks"
os.makedirs(CHUNK_DIR, exist_ok=True)

print(f"Total prompts : {len(prompts)}")
print(f"Chunks        : {N_CHUNKS} × {CHUNK_SIZE}")
print(f"This session  : {MY_CHUNKS}")
print(f"Batch size    : {BATCH}  |  n_shots: {INF_CONFIG['n_shots']}  |  encoder: {INF_CONFIG['encoder_model']}")
print(f"Chunk dir     : {CHUNK_DIR}\n")

model.enable_adapter_layers()
t0 = time.time()

for chunk_idx in MY_CHUNKS:
    chunk_path = f"{CHUNK_DIR}/chunk_{chunk_idx:02d}.csv"
    start = chunk_idx * CHUNK_SIZE
    end   = start + CHUNK_SIZE if chunk_idx < N_CHUNKS - 1 else len(prompts)
    expected = end - start

    if os.path.exists(chunk_path) and len(pd.read_csv(chunk_path)) == expected:
        print(f"[chunk {chunk_idx:02d}] already complete — skipping")
        continue

    chunk_prompts = prompts[start:end]
    chunk_ids     = ids[start:end]
    chunk_svgs    = []
    invalid_count = 0
    chunk_t0      = time.time()

    for i in range(0, len(chunk_prompts), BATCH):
        batch_svgs = generate_svg_batch(chunk_prompts[i : i + BATCH], batch_size=BATCH)
        for svg in batch_svgs:
            if not is_valid_svg(svg): invalid_count += 1
            chunk_svgs.append(svg)
        done    = min(i + BATCH, len(chunk_prompts))
        elapsed = time.time() - chunk_t0
        avg     = elapsed / done
        eta     = avg * (len(chunk_prompts) - done) / 60
        print(f"  [chunk {chunk_idx:02d} | {done:3d}/{len(chunk_prompts)}]  "
              f"avg={avg:.1f}s  invalid={invalid_count}  ETA={eta:.1f}min")

    pd.DataFrame({"id": chunk_ids, "svg": chunk_svgs}).to_csv(chunk_path, index=False)
    print(f"[chunk {chunk_idx:02d}] saved  (invalid={invalid_count}, "
          f"elapsed={( time.time()-t0)/60:.1f}min)\n")

print(f"Session done in {(time.time()-t0)/60:.1f} min.")

# ── Merge when all 10 chunks are ready ────────────────────────────────────────
print("\nChecking chunks …")
missing = []
for i in range(N_CHUNKS):
    path = f"{CHUNK_DIR}/chunk_{i:02d}.csv"
    start = i * CHUNK_SIZE
    end   = start + CHUNK_SIZE if i < N_CHUNKS - 1 else len(prompts)
    expected = end - start
    if not os.path.exists(path):
        missing.append(f"chunk_{i:02d} (missing)")
    else:
        n = len(pd.read_csv(path))
        if n != expected:
            missing.append(f"chunk_{i:02d} (has {n}, need {expected})")

if missing:
    print(f"Not ready to merge. Still waiting on: {missing}")
else:
    print("All chunks complete — merging …")
    chunks = [pd.read_csv(f"{CHUNK_DIR}/chunk_{i:02d}.csv") for i in range(N_CHUNKS)]
    sub_df = pd.concat(chunks, ignore_index=True)
    assert len(sub_df) == len(prompts), f"Row mismatch: {len(sub_df)} vs {len(prompts)}"
    sub_df.to_csv(INF_CONFIG["submission_path"], index=False)
    total_invalid = sum(not is_valid_svg(s) for s in sub_df["svg"])
    print(f"Saved   : {INF_CONFIG['submission_path']}")
    print(f"Rows    : {len(sub_df)}")
    print(f"Invalid : {total_invalid}")
    sub_df.head()

Total prompts : 1000
Chunks        : 10 × 100
This session  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Batch size    : 16  |  n_shots: 5  |  encoder: all-MiniLM-L6-v2
Chunk dir     : /content/drive/MyDrive/DL/midterm/inference_2B_1_chunks

  [chunk 00 |  16/100]  avg=14.0s  invalid=0  ETA=19.6min
  [chunk 00 |  32/100]  avg=15.0s  invalid=0  ETA=16.9min
  [chunk 00 |  48/100]  avg=15.7s  invalid=0  ETA=13.6min
  [chunk 00 |  64/100]  avg=15.7s  invalid=0  ETA=9.4min
  [chunk 00 |  80/100]  avg=15.9s  invalid=0  ETA=5.3min
  [chunk 00 |  96/100]  avg=16.0s  invalid=0  ETA=1.1min
  [chunk 00 | 100/100]  avg=16.9s  invalid=0  ETA=0.0min
[chunk 00] saved  (invalid=0, elapsed=28.1min)

  [chunk 01 |  16/100]  avg=17.3s  invalid=0  ETA=24.2min
  [chunk 01 |  32/100]  avg=16.0s  invalid=0  ETA=18.2min
  [chunk 01 |  48/100]  avg=16.3s  invalid=0  ETA=14.2min
  [chunk 01 |  64/100]  avg=16.3s  invalid=0  ETA=9.8min
  [chunk 01 |  80/100]  avg=16.4s  invalid=0  ETA=5.5min
  [chunk 01 |  96/100]  avg=16.6